# Rubin Observatory Data

<p>
    <img src="https://uwashington-astro300.github.io/A300_images/Rubin01.png" width = "500">
    <img src="https://uwashington-astro300.github.io/A300_images/Rubin02.png" width = "500">
</p>


Located on a mountaintop in Chile, NSF–DOE Vera C. Rubin Observatory is capturing the cosmos in exquisite detail. Using the largest camera ever built, Rubin is repeatedly scanning the sky for 10 years to create an ultra-wide, ultra-high-definition time-lapse record of our Universe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lsst.rsp import RSPDiscovery

from astropy.table import QTable, join

from astropy import units as u
from astropy import constants as const
from astropy.units import imperial
imperial.enable()

discovery = RSPDiscovery("dp2")
service = discovery.get_tap_client()

---

## Rubin is the most powerful asteroid discovery machine yet developed.

# Rubin [`SSObject` database](https://sdm-schemas.lsst.io/dp2.html#SSObject)

`SSObject` stores Solar System objects and derived parameters for moving celestial bodies. Some of the important parameters include the photometic magnitudes in each of the color filters of the Rubin camera.

<p>
    <img src="https://uwashington-astro300.github.io/A300_images/RubinFilters.jpg" width = "900">
</p>

### We will access data from the Rubin Observatory in much the same way we accessed Gaia data

### Using the ADQL language

### An Example

Let's get the `designation` and the absolute magnitude (`H`) in the g, r, i, and z filter of the first ten objects in the `SSObject` database

In [ ]:
my_query = """
SELECT TOP 10
designation, g_H, r_H, i_H, z_H
FROM dp2.SSObject
WHERE g_H IS NOT NULL
AND r_H IS NOT NULL
AND i_H IS NOT NULL
AND z_H IS NOT NULL
ORDER BY designation ASC
"""

In [ ]:
print(my_query)

### Send the query to the Rubin Server

In [ ]:
job = service.submit_job(my_query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

### Fetch the job results and put them in a `Qtable`.

In [ ]:
assert job.phase == 'COMPLETED'
result = job.fetch_result()
my_table = result.to_qtable()

In [ ]:
my_table

---

## Asteroid Colors

Much like stars, the color of an asteroids can give us clues to the composition of the object.


<p>
    <img src="https://uwashington-astro300.github.io/A300_images/Colors.jpg" width = "600">
</p>

This is an asteroid color-color diagram. The Y-axis is the (i-z) color, and the X-axis is a composite color called a-star and is defined as:

$$\Large
a^{\star} = 0.89(g-r) + 0.45(r-i) - 0.57\ \mathrm{mag}
$$

The position of the asteroid in the color-color diagram depends on the composition of the asteroid. 

- C-Type (carbonaceous asteroid) - The most common type of asteroid in our solar system, making up about 75% of all known asteroids. They are rich in carbon molecules, clay, silicate rocks, and water-bearing minerals.

- S-Type (stony asteroids) Made primarily of stony silicate materials mixed with nickel-iron metal. They make up about 17% of all known asteroids.

- V-Type (Vestoids) - Differentiated asteroids that are rich in pyroxene and basalt. Believed to be fragments blasted off the crust of the asteroid 4 Vesta. They make up roughly 6% of main-belt asteroids.

In [ ]:
my_table['a_star'] = ( 
    0.89 * (my_table['g_H'] - my_table['r_H']) +
    0.45 * (my_table['r_H'] - my_table['i_H']) - 0.57 * u.mag
)                          

In [ ]:
my_table['i_z'] = my_table['i_H'] - my_table['z_H']

In [ ]:
my_table

## Plotting: X-Y Plots

When plotting columns from these tables for X-Y plots
- **Use `.value` after the column name (i.e. `my_table['i_z'].value`)**

In [ ]:
fig, ax = plt.subplots(
    figsize = (8, 5), 
    constrained_layout = True
)

ax.set_xlim(-0.3, 0.3)
ax.set_ylim(-0.6, 0.6)

ax.set_xlabel('a_star',
              fontsize = 18)

ax.set_ylabel('i-z',
              fontsize = 18)

ax.vlines(0, -0.6, 0.6,
          color = 'LawnGreen',
          linewidth = 5,
          linestyle = '--')

ax.hlines(-0.10, 0, 0.3,
          color = 'LawnGreen',
          linewidth = 5,
          linestyle = '--')


ax.text(-0.28, 0.4,
       'C',
        color='HotPink',
        fontsize = 36)

ax.text(0.25, 0.4,
       'S',
        color='HotPink',
        fontsize = 36)

ax.text(0.25, -0.5,
       'V',
        color='HotPink',
        fontsize = 36)

ax.plot(my_table['a_star'].value, my_table['i_z'].value,
        color = 'MidnightBlue',
        marker = '*',
        markersize = 16,
        linestyle = 'None');

---
## Side Topic - Tuples

Tuples are closely related to arrays and lists, but has the important difference is they are immutable. 
- Tuples cannot be changed, added to, or removed from once created.
- Tuples are good for collections of data where preserving order is important.

In [ ]:
my_tuple = (4, 3, 2)

In [ ]:
my_tuple

In [ ]:
type(my_tuple)

### If you sort a `tuple` it gets turned into a `list`. Notice the `[]` delimiters

In [ ]:
sorted(my_tuple)

---
# [`mpc_orbits` database](https://sdm-schemas.lsst.io/dp2.html#mpc_orbits)

This is a database of solar system objects from the [Minor Planets Center](https://minorplanetcenter.net/). This database contains the orbital elements for all solar system objects.

Let's get some orbit elements for our objects that we queried from the `SSObject` database

- The first thing we need to do is to create a object list from the `designation` column of our table
- This object list needs to be a `tuple`
- `np.array` -> `list` -> `tuple`

In [ ]:
my_objects = tuple(my_table['designation'].tolist())

In [ ]:
my_objects

### We can then use `f-string` formatting to include this target list in the ADQL query

- `q` = perihelion distance (AU)
- `e` = orbital eccentricity
- `i` = orbital inclination (deg)

In [ ]:
my_mpc_query = f"""
SELECT designation, q, e, i
FROM dp2.mpc_orbits
WHERE designation IN {my_objects}
ORDER BY designation ASC
"""

In [ ]:
print(my_mpc_query)

In [ ]:
job = service.submit_job(my_mpc_query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

### Fetch the job results and assign to an astropy `result` table.

In [ ]:
assert job.phase == 'COMPLETED'
result = job.fetch_result()
my_other_table = result.to_qtable()

In [ ]:
my_other_table

## Add a column for the semi-major axis `a`

$$\Large
a = \frac{q}{1-e}
$$

In [ ]:
my_other_table['a'] = my_other_table['q'] / (1 - my_other_table['e'])

In [ ]:
my_other_table

# Joining Tables - Inner Join

An Inner join combines rows from two or more tables based on a related column, returning only the rows that have matching values in both tables

- Our two tables `my_table` and `my_other_table` share a column: `designation`.
- We can join the rows of theses tables that have the same `designation` value.
- Ignore the initial warning!

In [ ]:
my_big_table = join(my_table, my_other_table, 
                    keys = 'designation', 
                    join_type ='inner'
                   )

In [ ]:
my_big_table

## Plotting - Histograms

When plotting columns from these tables for histograms:
- **Use `.data` after the column name (i.e. `my_table['a'].data`)**

In [ ]:
my_bins = np.arange(2.1,3.2,0.1)
my_bins

In [ ]:
fig, ax = plt.subplots(
    figsize = (8, 5), 
    constrained_layout = True
)

ax.set_xlabel('semi-major-axis (AU)',
              fontsize = 18)

ax.set_ylabel('Number',
              fontsize = 18)


ax.hist(my_big_table['a'].data,
        bins = my_bins,
        histtype = 'stepfilled',
        facecolor = 'MediumOrchid',
        label = "Objects")


ax.legend(loc=0, shadow=True);

### Tons of other examples of exploring Rubin data can be found in the `Tutorials` tab above. 

### There a lots for very cool tutorials in the `Tutorials` -> `DP2` -> `300_Science_demo` sections!